# RAG Bot 

The RAG bot system to answer the question about Ethnicity in Thailand.

## Setting Up   

### Install Libraries & Dependencies  

In [51]:
%pip install python-dotenv   

Note: you may need to restart the kernel to use updated packages.


### Import Libraries & Dependencies  

In [52]:
import os, re, json  
import time  
import requests  
from dotenv import load_dotenv   

### Setting Environments  

In [53]:
load_dotenv()

True

In [54]:
api_key = os.getenv("THAILLM_API_KEY") 

if api_key:
  print(f"[LOG] Success: Get API Key success.")
else:
  print(f"[LOG] Fail: Cannot Get API Key.")  

[LOG] Success: Get API Key success.


## LLM Implementation  

In [55]:
def ask(messages: str, model="typhoon", max_retries=5):
  url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"  
  headers = {
    "Content-Type": "application/json",
    "apikey": api_key  
  }
  payload = {
    "model": "/model",
    "messages": messages,
    "max_tokens": 2024,
    "temperature": 0 
  }

  for attempt in range(max_retries):
    try:
      resp = requests.post(url=url, headers=headers, json=payload, timeout=120)

      if resp.status_code == 429:
        wait = min(2 ** attempt, 30)
        print(f"Rate Limit, wating {wait} sec(s).")
        time.sleep(wait)
        continue 
      
      resp.raise_for_status()
      text_answer = resp.json()["choices"][0]["message"]["content"].strip() 
      clean_answer = re.sub(r"<think>.*?</think>", "", text_answer, flags=re.DOTALL).strip()
      return clean_answer    
    except requests.exceptions.RequestException as e:
      wait = 2 ** attempt
      print(f"Error: {e}, Retrying in {wait} sec(s).")
      time.sleep(wait)
    
  return None  


In [56]:
def qa(question: str):
  prompt = {"role": "user", "content": question}

  answer_typhoon = ask([prompt], model="typhoon")
  answer_kbtg = ask([prompt], model="kbtg")
  answer_openthaigpt = ask([prompt], model="openthaigpt")
  answer_pathumma = ask([prompt], model="pathumma")

  # JSON Format
  # answer = {
  #   "Typhoon": f"[RAG BOT (Typhoon)] {answer_typhoon}",
  #   "KBTG": f"[RAG BOT (KBTG)] {answer_kbtg}",
  #   "OpenThaiGPT": f"[RAG BOT (OpenThaiGPT)] {answer_openthaigpt}",
  #   "Pathumma": f"[RAG BOT (Pathumma)] {answer_pathumma}"
  # }

  # Markdown Format   
  answer = f"""# Question
{question}

# Answer

## Typhoon
{answer_typhoon}

## KBTG
{answer_kbtg}

## OpenThaiGPT
{answer_openthaigpt}

## Pathumma
{answer_pathumma} 
"""

  return answer   

### LLM Implementation Testing  

In [57]:
def report(question: str, filename: str):
  # question = "คนไทยเชื้อสายจีนในเมืองหาดใหญ่ ส่วนมากเป็นคนเชื้อสายจีนสายไหน"
  result = qa(question)

  # output = {
  #   "question": question,
  #   "answer": test_response_01
  # }

  output_path = "../output/answer/markdown"  

  if filename:
    file_name = filename
  else:
    file_name = "_".join(question.split(" "))

  with open(f"{output_path}/{file_name}.md", "w", encoding="utf-8") as file:
    file.write(result)  

In [58]:
report("คนไทยเชื้อสายจีนไหหลำในไทย อาศัยอยู่กันเยอะในจังหวัดอะไรในประเทศไทยบ้าง", filename="QA_Hainanese")  